# UKB tofu DUAC 06 Sulzbach

## 0. Definitions

### 0.1 Libraries

In [1]:
# IO
import glob
import os
import datetime
import time

# Import
import xlrd
import xlsxwriter
import openpyxl
from openpyxl import load_workbook

# Data Handling
import pandas as pd
import numpy as np
import math

# Stats
from tableone import TableOne
import scipy.stats as st
from scipy.stats import sem
import pingouin as pg

# Figures
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

In [2]:
# Date
# using now() to get current time
dt = datetime.datetime.now()
# dt.year, dt.month, dt.day, dt.hour, dt.minute, dt.second, dt.microsecond
date = '%04d%02d%02d' % (dt.year, dt.month, dt.day)
date

'20260914'

### 0.3 Stat Funtions

In [3]:
def p2s(p):
    if p < 0.001:
        si = '***'
    elif p < 0.01: 
        si = '**'
    elif p < 0.05:
        si = '*'
    else: 
        si = 'n.s.'
    return si

In [4]:
# newly framed into df as output
# 2025 11 10 fk

def twoByTwo (a,b,c,d):

    #            Y   N    Total
    # group 1:   a   b    t1
    # group 2:   c   d    t2
    # totals:    ty  tn   nn
    
    t1 = a+b
    t2 = c+d
    ty = a+c
    tn = b+d
    nn = a+b+c+d
    
    chi2 = nn * (a*d - b*c)**2 / (t1*t2*ty*tn)
    phi = np.sqrt(chi2/nn)
    cd = (phi-0.1)*1.5 + 0.2
    sens = a/t1
    spec = d/t2
    oddsr = (a/b)/(c/d)
    riskr = (a/t1) / (c/t2)
    corr = (a+d)/nn
    ppsc = a / (a+c)
    p = 1 - st.chi2.cdf(chi2, 1)

    resp = pd.DataFrame(columns = ['chi2', 'phi', 'cd', 
                                   'p', 'sign', 'ps', 
                                   'sens', 'spec', 'correct', 
                                   'oddsr', 'riskr','ppsc'])
    resp.loc[0] = [np.round(chi2,3), np.round(phi,2), np.round(cd,2), 
                   np.round(p,4), p2s(p), "<0.001" if p<0.001 else "%0.3f"%p,
                   np.round(sens,2), np.round(spec,2), np.round(corr,2), 
                   np.round(oddsr,3), np.round(riskr,3), np.round(ppsc,2)]
    # Old returns
    #return [[a,b,c,d],
    #       ['X² = %0.2f, p = %0.4f, phi = %0.2f, d = %0.2f'%(chi2,p,phi, cd)],
    #       ['sens = %0.2f, spec = %0.4f, corr = %0.2d, ppsc = %0.2f'%(sens, spec,corr, ppsc)]]
    #    return {'descriptive' : msd,
    #         'stats': res, 
    #         'summary': res.replace(pd.NA, "").to_string(index=False), 
    #         'Factors': res.Factors.values,#
    #         'FactorNames' : res.FactorNames.values,
    #         'SSQ' : res.SSQ.values,
    #         'df' : res.df.values,
    #         'SD2' : res.S2.values,
    #         'F' : res.F.values,
    #         'p' : res.p.values}
    return resp


In [6]:
twoByTwo(55, 45, 51, 49)

,chi2,phi,cd,p,sign,ps,sens,spec,correct,oddsr,riskr,ppsc
0,0.321,0.04,0.11,0.5709,n.s.,0.571,0.55,0.49,0.52,1.174,1.078,0.52


### 0.4 Helper Functions

In [6]:
# neu mit concat statt append: September 2022
def hat(df, x=5):
    return pd.concat([df.head(x),df.tail(x)])
setattr(pd.DataFrame,'hat',hat)

### 0.5 Medical and Psychological Scores

### 0.9 Main Paths

In [1]:
import os

mainPath = "/workspace"

inPath = os.path.join(mainPath, "data", "raw")
outPath = os.path.join(mainPath, "results")

os.makedirs(outPath, exist_ok=True)

print("Input:", inPath)
print("Output:", outPath)

Input: /workspace\data\raw
Output: /workspace\results


## 1. Import

### 1.1 Import the overall (BL+FU) cleaned data set with visual acuity measures

In [6]:
main_file = os.path.join(
    inPath,
    "tofu DQA 2026Q2 cleaned DS+VA.xlsx"
)

dfA = pd.read_excel(main_file)

print("Dataset shape:", dfA.shape)

FileNotFoundError: [Errno 2] No such file or directory: '/workspace\\data\\raw\\tofu DQA 2026Q2 cleaned DS+VA.xlsx'

In [8]:
os.chdir(inPath)
fileNames = sorted(glob.glob("*.xlsx"))
print(fileNames)

['tofu DQA 2026Q2 cleaned DS+VA.xlsx']


In [9]:
dfA = pd.read_excel(fileNames[0])
# low_memory = Warning for mixed data types (mostly text and nans) 
dfA.hat(2)

,record_id,redcap_repeat_instrument,redcap_repeat_instance,trialprtstat_yn,stchng_inc_yn,stchng_inc_dat,stchng_inc_dat_clc1,stchng_inftransf_yn,site_no,pat_no,...,dtinclicx,dtvisic,dtvisicx,vLum,vRum,vLcm,vRcm,vLbm,vRbm,vBbm
0,1,NaN,NaN,1.0,NaN,NaN,88.0,NaN,1,377,...,0.0,0.0,0.0,NaN,NaN,0.39794,0.49485,0.4,0.49,0.4
1,10-1,NaN,NaN,1.0,NaN,NaN,88.0,NaN,4,1,...,0.0,0.0,0.0,NaN,0.096910,NaN,NaN,NaN,0.10,0.1
8310,9-8,follow_up,1.0,NaN,NaN,NaN,NaN,NaN,3,8,...,NaN,NaN,NaN,0.00000,NaN,NaN,0.09691,0.0,0.10,0.0
8311,9-8,follow_up,2.0,NaN,NaN,NaN,NaN,NaN,3,8,...,NaN,NaN,NaN,0.09691,0.221849,0.00000,0.00000,0.0,0.00,0.0


### 1.2 Variable Lists

In [175]:
# Standard Data
stVarList = ['Patient_ID', 'dtFirstDiagn',
       'dtAge', 'dtAgeG', 'uv_aff_eye',
             'vis_no', 'vis_type','dtVisDays', 
       'uvLoc', 'uvLocS',
       'vLum', 'vRum', 'vLcm',
       'vRcm', 'vLbm', 'vRbm', 'vBbm']

In [176]:
# Interventions (Tasks 1 + 2)
intervListRL = ['in_ctr_yn_r', 'in_ctr_yn_l', 'in_vich_yn_r', 'in_vich_yn_l', 'in_glc_yn_r', 'in_glc_yn_l', 
               'in_encl_yn_r', 'in_encl_yn_l', 'in_ivo_anti_r', 'in_ivo_anti_l', 'in_ivo_stero_r', 'in_ivo_stero_l']
intervListRLnova = ['ctr_r', 'ctr_l', 'vich_r', 'vich_l', 'glc_r', 'glc_l', 
               'encl_r', 'encl_l', 'ivo_anti_r', 'ivo_anti_l', 'ivo_stero_r', 'ivo_stero_l']
intervList = ['ctr', 'vich', 'glc', 'encl',  'ivo_anti', 'ivo_stero']
intervListEP = intervList + ['EP1', 'EPs1']
dfA[intervListRL].hat(2)

,in_ctr_yn_r,in_ctr_yn_l,in_vich_yn_r,in_vich_yn_l,in_glc_yn_r,in_glc_yn_l,in_encl_yn_r,in_encl_yn_l,in_ivo_anti_r,in_ivo_anti_l,in_ivo_stero_r,in_ivo_stero_l
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,1.0,NaN,0.0,NaN,0.0,NaN,1.0,NaN,NaN,NaN,NaN
8310,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8311,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 1.3 PatLists

In [177]:
patListR = dfA[dfA.uv_aﬀ_eye.isin([1,2])].Patient_ID.unique()
patListL = dfA[dfA.uv_aﬀ_eye.isin([1,3])].Patient_ID.unique()
patListP = dfA[dfA.uv_aﬀ_eye.isin([1,2, 3])].Patient_ID.unique()

print("Affected Eye: L, R, B = ",len(patListL), len(patListR), len(patListB))

Affected Eye: L, R, B =  1472 1501 1310


### 1.4 Copying to dfB with required variables only

In [178]:
dfB = dfA[stVarList + intervListRL].copy()
dfB.columns = stVarList + intervListRLnova

dfB.columns

Index(['Patient_ID', 'dtFirstDiagn', 'dtAge', 'dtAgeG', 'uv_aff_eye', 'vis_no',
       'vis_type', 'dtVisDays', 'uvLoc', 'uvLocS', 'vLum', 'vRum', 'vLcm',
       'vRcm', 'vLbm', 'vRbm', 'vBbm', 'ctr_r', 'ctr_l', 'vich_r', 'vich_l',
       'glc_r', 'glc_l', 'encl_r', 'encl_l', 'ivo_anti_r', 'ivo_anti_l',
       'ivo_stero_r', 'ivo_stero_l'],
      dtype='object')

### ...

## 2. Transformations

### 2.1 Endpoints on patient level

In [179]:
for v in intervList :
    dfB[v+"_r"] = dfB[v+"_r"].replace(99.3, 0).fillna(0)
    dfB[v+"_l"] = dfB[v+"_l"].replace(99.3, 0).fillna(0)
    dfB[v+"_e"] = dfB[[v+"_r", v+"_l"]].replace(99.3, 0).sum(axis = 1)
    dfB[v+"_p"] = dfB[[v+"_r", v+"_l"]].replace(99.3, 0).max(axis = 1)

### 2.2 Primary Endpoint

In [180]:
dfB['EP1_r'] = dfB[['ctr_r', 'vich_r', 'glc_r', 'encl_r',  'ivo_anti_r', 'ivo_stero_r', ]].max(axis = 1)
dfB['EP1_l'] = dfB[['ctr_l', 'vich_l', 'glc_l', 'encl_l',  'ivo_anti_l', 'ivo_stero_l', ]].max(axis = 1)
dfB['EP1_e'] = dfB['EP1_l'] + dfB['EP1_r']
dfB['EP1_p'] = dfB[['ctr_p', 'vich_p', 'glc_p', 'encl_p',  'ivo_anti_p', 'ivo_stero_p', ]].max(axis = 1)

In [181]:
dfB['ctr_r'].value_counts()

ctr_r
0.0    7878
1.0     434
Name: count, dtype: int64

### 2.3 Sensitivity 1

In [182]:
# only CAT (ctr), PPV (vich), GLC (glc) and Enukl (enci)
dfB['EPs1_r'] = dfB[['ctr_r', 'vich_r', 'glc_r', 'encl_r']].max(axis = 1)
dfB['EPs1_l'] = dfB[['ctr_l', 'vich_l', 'glc_l', 'encl_l' ]].max(axis = 1)
dfB['EPs1_e'] = dfB['EPs1_l'] + dfB['EPs1_r']
dfB['EPs1_p'] = dfB[['ctr_p', 'vich_p', 'glc_p', 'encl_p']].max(axis = 1)

### 2.4 dfC First event ... for Frage 2 (Time-to-Event)

In [208]:
# For the time being only the first event and only if the eye is affected.
# For more events, copy code and define dfCn (and rename this into C1 ??) 
dfC = pd.DataFrame(columns = ["PID", "PIDE", "affected", "Parameter", "Event", "Day"])

In [209]:
for pid in patListP [:4]: 
    affected = dfB[(dfB.Patient_ID == pid) & (dfB.vis_type == "BL")].uv_aﬀ_eye.iloc[0]
    affEyes = ['l', 'r'] if affected == 1 else ['r'] if affected == 2 else ['l']
    print(pid, affected, affEyes)
    dfb = dfB[(dfB.Patient_ID == p) & (dfB.vis_type == "FU")]
    vn = dfb.shape[0] 
    
    if vn > 0 :
        for rl in ['l', 'r'] :
            pide = str(pid) + rl
            lastvis = dfb.iloc[-1:].dtVisDays.values[0]
            for interv in intervListEP :
                en = int(dfb[interv+"_"+rl].sum())
                if en == 0 :
                    dfC.loc[dfC.shape[0]] = [pid, pide, affected, interv, 0, lastvis]
                else : 
                    dfbs = dfb[dfb[interv+"_"+rl] == 1]
                    for e in range(en):
                        days =  dfbs.dtVisDays.values
                        dfC.loc[dfC.shape[0]] = [pid, pide, rl in affEyes, interv, e+1, days[e]]
        
    

37888 1.0 ['l', 'r']
31414 2.0 ['r']
72604 1.0 ['l', 'r']
58898 3.0 ['l']


In [211]:
dfC1 = dfC[dfC.Event.isin([0,1])]

### ...

## 3. Analyses

In [97]:
dfB0 = dfB[dfB.vis_type == 'BL']

In [98]:
dfB0.shape[0]

1662

In [100]:
NR = len(patListR)
NL = len(patListL)
NP = len(patListP)
NE = int(nr + nl)
print(NL, NR, NE, NP)

1472 1501 497 1662


### 3.1 Question 1

In [101]:
# ['ctr', 'vich', 'glc', 'encl',  'ivo_anti', 'ivo_stero']

In [102]:
# result = binomtest(k=7, n=50, p=0.1)
from scipy.stats import binomtest
binomtest(k=7, n=50).proportion_ci(method = "wilson")


ConfidenceInterval(low=np.float64(0.06950833427016292), high=np.float64(0.26186193710585537))

In [144]:
dfT1 = pd.DataFrame(columns = ['Subgroup', 'Period', 'Variable', 'Measure', 'Left', 'Right', 'Eye', 'Patient'])
for sub in range(10):
    if sub == 0:       # all pat
        dfs = dfB
        subN = "AllPatients"
    elif sub in [1,2,3,4]:     # uveitis loc
        dfs = dfB[dfB.uvLoc == sub]
        subN = ['Uveitis', 'Uv.Interm.', 'Uv.Post', 'Panuv.','Ret.Vasc.'  ][sub]
    elif sub == 5:
        dfs = dfB[dfB.dtAge <= 40]
        subN = "Age 18-40 y"
    elif sub == 6:
        dfs = dfB[dfB.dtAge <= 40]
        subN = "Age 41-60 y"
    elif sub == 7:
        dfs = dfB[dfB.dtAge <= 40]
        subN = "Age 61- y"
    elif sub == 8:
        dfs = dfB[dfB.vBbm < 0.3]
        subN = "VA < 0.3 logMAR"
    elif sub == 9:
        dfs = dfB[dfB.vBbm >= 0.3]
        subN = "VA >= 0.3 logMAR"

    NR = dfs[(dfs.vis_type == 'BL') & (dfs.Patient_ID.isin(patListR))].shape[0]
    NL = dfs[(dfs.vis_type == 'BL') & (dfs.Patient_ID.isin(patListL))].shape[0]
    NP = dfs[(dfs.vis_type == 'BL') & (dfs.Patient_ID.isin(patListP))].shape[0]
    NE = NL + NR 
    
    dfT1.loc[dfT1.shape[0]] = [subN, '','Total', 'N (%)', "%d (100%%)"%NL, "%d (100%%)"%NR, "%d (100%%)"%NE, "%d (100%%)"%NP]
        
        
    for tp in ['BL', 'FU']:
        dfb = dfs[dfs.vis_type == tp]
    
        for v in intervList + ['EP1', 'EPs1']: # v for variable
            nl = dfb[v+"_l"].sum().astype(int)
            cl0, cl1 = binomtest(k=nl, n=NL).proportion_ci(method = "wilson")
            
            nr = dfb[v+"_r"].sum().astype(int)
            cr0, cr1 = binomtest(k=nr, n=NR).proportion_ci(method = "wilson")
            
            ne = dfb[v+"_e"].sum().astype(int)
            ce0, ce1 = binomtest(k=ne, n=NE).proportion_ci(method = "wilson")
            
            np = dfb[v+"_p"].sum().astype(int)
            if np < NP :
                cp0, cp1 = binomtest(k=np, n=NP).proportion_ci(method = "wilson")
            else :
                cp0 = cp1 = 1
            # print(nl, cl0, cl1)
            dfT1.loc[dfT1.shape[0]] = [subN, tp, v, 'N (%)', 
                                       "%d, %.1f%% [%.1f – %.1f]"%(nl, nl/NL*100, cl0*100, cl1*100), 
                                       "%d, %.1f%% [%.1f – %.1f]"%(nr, nr/NR*100, cr0*100, cr1*100), 
                                       "%d, %.1f%% [%.1f – %.1f]"%(ne, ne/NE*100, ce0*100, ce1*100), 
                                       "%d, %.1f%% [%.1f – %.1f]"%(np, np/NP*100, cp0*100, cp1*100) 
                                        ]

In [145]:
dfT1.to_excel("tofu_DUAC06_Question01_Table.xlsx", index = False)

In [146]:
dfT1.iloc[-15:, :]

,Subgroup,Period,Variable,Measure,Left,Right,Eye,Patient
155,VA >= 0.3 logMAR,BL,vich,N (%),"21, 11.4% [7.6 – 16.8]","21, 10.9% [7.3 – 16.1]","42, 11.2% [8.4 – 14.8]","31, 15.9% [11.4 – 21.7]"
156,VA >= 0.3 logMAR,BL,glc,N (%),"6, 3.3% [1.5 – 6.9]","6, 3.1% [1.4 – 6.6]","12, 3.2% [1.8 – 5.5]","8, 4.1% [2.1 – 7.9]"
157,VA >= 0.3 logMAR,BL,encl,N (%),"1, 0.5% [0.1 – 3.0]","1, 0.5% [0.1 – 2.9]","2, 0.5% [0.1 – 1.9]","2, 1.0% [0.3 – 3.7]"
158,VA >= 0.3 logMAR,BL,ivo_anti,N (%),"14, 7.6% [4.6 – 12.4]","17, 8.9% [5.6 – 13.7]","31, 8.2% [5.9 – 11.5]","24, 12.3% [8.4 – 17.7]"
159,VA >= 0.3 logMAR,BL,ivo_stero,N (%),"36, 19.6% [14.5 – 25.9]","40, 20.8% [15.7 – 27.1]","76, 20.2% [16.5 – 24.6]","48, 24.6% [19.1 – 31.1]"
160,VA >= 0.3 logMAR,BL,EP1,N (%),"89, 48.4% [41.3 – 55.5]","91, 47.4% [40.5 – 54.4]","180, 47.9% [42.9 – 52.9]","103, 52.8% [45.8 – 59.7]"
161,VA >= 0.3 logMAR,BL,EPs1,N (%),"71, 38.6% [31.9 – 45.8]","73, 38.0% [31.5 – 45.1]","144, 38.3% [33.5 – 43.3]","82, 42.1% [35.3 – 49.1]"
162,VA >= 0.3 logMAR,FU,ctr,N (%),"37, 20.1% [15.0 – 26.5]","40, 20.8% [15.7 – 27.1]","77, 20.5% [16.7 – 24.8]","53, 27.2% [21.4 – 33.8]"
163,VA >= 0.3 logMAR,FU,vich,N (%),"25, 13.6% [9.4 – 19.3]","23, 12.0% [8.1 – 17.3]","48, 12.8% [9.8 – 16.5]","41, 21.0% [15.9 – 27.3]"
164,VA >= 0.3 logMAR,FU,glc,N (%),"6, 3.3% [1.5 – 6.9]","3, 1.6% [0.5 – 4.5]","9, 2.4% [1.3 – 4.5]","9, 4.6% [2.4 – 8.5]"


In [117]:
nr

np.int64(137)

In [118]:
nl

np.int64(130)

In [119]:
ne

np.int64(267)

In [120]:
np

np.int64(198)

In [121]:
NR

192

In [122]:
NL

184

In [123]:
NE

376

In [124]:
NP

195